# Tech Challenge Fase 2
## 05.4 — Data Quality Dashboard

Consolida os resultados de qualidade das camadas Bronze, Silver e Gold em uma visão executiva única, pronta para Databricks e Power BI.

Saídas:

```text
logs/data_quality/dashboard/details
logs/data_quality/dashboard/summary
logs/data_quality/dashboard/kpis
logs/data_quality/dashboard/layer_status
logs/data_quality/dashboard/dataset_ranking
gold/exports_powerbi/POWERBI_QUALITY_DASHBOARD
```

## 1. Imports

In [0]:
import json

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Configuração

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

LOG_PATH = config["paths"]["log_path"]
GOLD_PATH = config["paths"]["gold_path"]
EXECUTION_DATE = config["project"]["execution_date"]

QUALITY_ROOT_PATH = f"{LOG_PATH}/data_quality"
QUALITY_DASHBOARD_PATH = f"{QUALITY_ROOT_PATH}/dashboard"

DETAILS_PATHS = {
    "bronze": f"{QUALITY_ROOT_PATH}/bronze/details/execution_date={EXECUTION_DATE}",
    "silver": f"{QUALITY_ROOT_PATH}/silver/details/execution_date={EXECUTION_DATE}",
    "gold": f"{QUALITY_ROOT_PATH}/gold/details/execution_date={EXECUTION_DATE}"
}

SUMMARY_PATHS = {
    "bronze": f"{QUALITY_ROOT_PATH}/bronze/summary/execution_date={EXECUTION_DATE}",
    "silver": f"{QUALITY_ROOT_PATH}/silver/summary/execution_date={EXECUTION_DATE}",
    "gold": f"{QUALITY_ROOT_PATH}/gold/summary/execution_date={EXECUTION_DATE}"
}

POWERBI_EXPORT_PATH = f"{GOLD_PATH}/exports_powerbi/quality_dashboard"

for path in [
    QUALITY_DASHBOARD_PATH,
    f"{QUALITY_DASHBOARD_PATH}/details",
    f"{QUALITY_DASHBOARD_PATH}/summary",
    f"{QUALITY_DASHBOARD_PATH}/kpis",
    f"{QUALITY_DASHBOARD_PATH}/layer_status",
    f"{QUALITY_DASHBOARD_PATH}/dataset_ranking",
    POWERBI_EXPORT_PATH
]:
    dbutils.fs.mkdirs(path)

print("EXECUTION_DATE:", EXECUTION_DATE)

## 3. Funções auxiliares

In [0]:
def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def read_required_parquet(path, name):
    if not path_exists(path):
        raise FileNotFoundError(
            f"Resultado de qualidade ausente para {name}: {path}"
        )

    return spark.read.parquet(path)

## 4. Consolidação dos resultados detalhados

In [0]:
detail_frames = []

for layer, path in DETAILS_PATHS.items():
    df = read_required_parquet(
        path,
        f"detalhes {layer}"
    )

    detail_frames.append(df)

df_quality_details = detail_frames[0]

for df in detail_frames[1:]:
    df_quality_details = (
        df_quality_details
        .unionByName(
            df,
            allowMissingColumns=True
        )
    )

df_quality_details = (
    df_quality_details
    .withColumn(
        "status_order",
        F.when(
            F.col("status") == "REPROVADO",
            1
        )
        .when(
            F.col("status") == "ATENCAO",
            2
        )
        .otherwise(3)
    )
)

display(
    df_quality_details
    .orderBy(
        "status_order",
        "layer",
        "dataset",
        "ano",
        "rule_id"
    )
)

## 5. Consolidação dos resumos

In [0]:
summary_frames = []

for layer, path in SUMMARY_PATHS.items():
    df = read_required_parquet(
        path,
        f"resumo {layer}"
    )

    summary_frames.append(df)

df_quality_summary = summary_frames[0]

for df in summary_frames[1:]:
    df_quality_summary = (
        df_quality_summary
        .unionByName(
            df,
            allowMissingColumns=True
        )
    )

display(
    df_quality_summary
    .orderBy(
        "layer",
        "dataset",
        "ano"
    )
)

## 6. KPIs executivos

In [0]:
df_quality_kpis = (
    df_quality_details
    .agg(
        F.count("*").alias("total_rules"),
        F.sum(
            F.when(
                F.col("status") == "APROVADO",
                1
            ).otherwise(0)
        ).alias("approved_rules"),
        F.sum(
            F.when(
                F.col("status") == "ATENCAO",
                1
            ).otherwise(0)
        ).alias("warning_rules"),
        F.sum(
            F.when(
                F.col("status") == "REPROVADO",
                1
            ).otherwise(0)
        ).alias("failed_rules"),
        F.sum(
            F.when(
                (F.col("status") == "REPROVADO")
                & F.col("blocking"),
                1
            ).otherwise(0)
        ).alias("blocking_failed_rules"),
        F.sum(
            F.col("invalid_records")
        ).alias("total_invalid_records")
    )
    .withColumn(
        "quality_score_percent",
        F.round(
            F.col("approved_rules")
            / F.col("total_rules")
            * 100,
            2
        )
    )
    .withColumn(
        "platform_status",
        F.when(
            F.col("blocking_failed_rules") > 0,
            "REPROVADO"
        )
        .when(
            F.col("failed_rules") > 0,
            "ATENCAO"
        )
        .otherwise("APROVADO")
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(df_quality_kpis)

## 7. Status por camada

In [0]:
df_quality_layer_status = (
    df_quality_details
    .groupBy("layer")
    .agg(
        F.count("*").alias("total_rules"),
        F.sum(
            F.when(
                F.col("status") == "APROVADO",
                1
            ).otherwise(0)
        ).alias("approved_rules"),
        F.sum(
            F.when(
                F.col("status") == "ATENCAO",
                1
            ).otherwise(0)
        ).alias("warning_rules"),
        F.sum(
            F.when(
                F.col("status") == "REPROVADO",
                1
            ).otherwise(0)
        ).alias("failed_rules"),
        F.sum(
            F.when(
                (F.col("status") == "REPROVADO")
                & F.col("blocking"),
                1
            ).otherwise(0)
        ).alias("blocking_failed_rules"),
        F.sum(
            F.col("invalid_records")
        ).alias("invalid_records")
    )
    .withColumn(
        "quality_score_percent",
        F.round(
            F.col("approved_rules")
            / F.col("total_rules")
            * 100,
            2
        )
    )
    .withColumn(
        "layer_status",
        F.when(
            F.col("blocking_failed_rules") > 0,
            "REPROVADO"
        )
        .when(
            F.col("failed_rules") > 0,
            "ATENCAO"
        )
        .otherwise("APROVADO")
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_quality_layer_status
    .orderBy("layer")
)

## 8. Ranking por dataset

In [0]:
df_quality_dataset_ranking = (
    df_quality_summary
    .withColumn(
        "ranking_quality",
        F.dense_rank().over(
            Window.orderBy(
                F.col(
                    "quality_score_percent"
                ).desc(),
                F.col(
                    "total_invalid_records"
                ).asc()
            )
        )
    )
    .withColumn(
        "execution_date",
        F.lit(EXECUTION_DATE)
    )
)

display(
    df_quality_dataset_ranking
    .orderBy(
        "ranking_quality",
        "layer",
        "dataset",
        "ano"
    )
)

## 9. Regras críticas

In [0]:
df_critical_rules = (
    df_quality_details
    .filter(
        (F.col("severity") == "CRITICAL")
        | F.col("blocking")
    )
    .select(
        "layer",
        "dataset",
        "ano",
        "rule_id",
        "rule_name",
        "severity",
        "blocking",
        "status",
        "records_evaluated",
        "invalid_records",
        "invalid_percent",
        "message",
        "status_order"
    )
    .orderBy(
        "status_order",
        "layer",
        "dataset",
        "ano"
    )
)

if df_critical_rules.count() == 0:
    print("Nenhuma regra crítica configurada.")
else:
    display(
        df_critical_rules
        .drop("status_order")
    )

## 10. Persistência das bases do dashboard

In [0]:
outputs = [
    (
        df_quality_details,
        f"{QUALITY_DASHBOARD_PATH}/details/execution_date={EXECUTION_DATE}"
    ),
    (
        df_quality_summary,
        f"{QUALITY_DASHBOARD_PATH}/summary/execution_date={EXECUTION_DATE}"
    ),
    (
        df_quality_kpis,
        f"{QUALITY_DASHBOARD_PATH}/kpis/execution_date={EXECUTION_DATE}"
    ),
    (
        df_quality_layer_status,
        f"{QUALITY_DASHBOARD_PATH}/layer_status/execution_date={EXECUTION_DATE}"
    ),
    (
        df_quality_dataset_ranking,
        f"{QUALITY_DASHBOARD_PATH}/dataset_ranking/execution_date={EXECUTION_DATE}"
    )
]

for df, path in outputs:
    (
        df
        .coalesce(1)
        .write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(path)
    )

print("Bases do dashboard persistidas com sucesso.")

## 11. Exportação para Power BI

In [0]:
(
    df_quality_details
    .drop("status_order")
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .csv(POWERBI_EXPORT_PATH)
)

print("Exportação Power BI salva em:")
print(POWERBI_EXPORT_PATH)

## 12. Checklist final

In [0]:
blocking_failures = (
    df_quality_details
    .filter(
        (F.col("status") == "REPROVADO")
        & F.col("blocking")
    )
)

blocking_failure_count = (
    blocking_failures.count()
)

if blocking_failure_count > 0:
    display(
        blocking_failures
        .orderBy(
            "layer",
            "dataset",
            "ano",
            "rule_id"
        )
    )

    raise Exception(
        f"O Dashboard identificou "
        f"{blocking_failure_count} regras "
        f"bloqueantes reprovadas."
    )

print("Data Quality Dashboard concluído com sucesso.")
print("A plataforma está liberada para consumo analítico.")

## Resultado esperado

```text
logs/data_quality/dashboard/details/
logs/data_quality/dashboard/summary/
logs/data_quality/dashboard/kpis/
logs/data_quality/dashboard/layer_status/
logs/data_quality/dashboard/dataset_ranking/

gold/exports_powerbi/quality_dashboard/
```

Próxima etapa:

```text
06_monitoring
```